# Ultimate Production Ensemble Pipeline (Optimized)

This notebook implements the definitive, production-ready architecture for the Speech Emotion Recognition (SER) project. The execution bottlenecks and naive grid searches have been replaced with continuous mathematical optimization and strict early-stopping protocols.

### Architectural Masterpieces:
1. **Full Acoustic Integrity (KNN Imputed)**: Dynamically ingests all 48 features. Uses K-Nearest Neighbors imputation to preserve acoustic feature covariance instead of flattening data with median values.
2. **Soft-Voting Ensemble**: Blends XGBoost, LightGBM, and CatBoost using probabilities.
3. **Out-of-Fold (OOF) Weight Optimization (SLSQP)**: Uses 3-Fold Stratified Cross-Validation with early stopping to prevent fold overfitting. Applies Sequential Least SQuares Programming to mathematically prove the optimal continuous blending weights.
4. **Deployment Serialization**: Automatically exports the trained models, the data scaler, the label encoder, and the optimized weights directly to disk.

In [1]:
# ============================================================
# 1. Environment Setup & Library Imports
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import joblib
import warnings
import torch
from scipy.optimize import minimize

# Safely import boosting libraries
try: import xgboost as xgb
except ImportError: pass
try: import lightgbm as lgb
except ImportError: pass
try: from catboost import CatBoostClassifier
except ImportError: pass

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA Available (PyTorch):", CUDA_AVAILABLE)

CUDA Available (PyTorch): False


### 2. Data Ingestion & Preprocessing
We dynamically locate `all_emotions.csv`. Pandas intrinsically handles NA variations on load. Missing features are imputed using `KNNImputer` to respect the natural clustering of acoustic data.

In [2]:
# Resolve Paths
data_path = os.path.join('dataset', 'all_emotions.csv')
if not os.path.exists(data_path):
    data_path = 'all_emotions.csv' # Fallback

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path, na_values=["", " ", "nan", "NaN"])

# Detect Target Column
target_col = "label" if "label" in df.columns else "Emotion"
if target_col not in df.columns:
    target_col = df.columns[-1]

# Clean Missing Labels
df_cleaned = df.dropna(subset=[target_col]).copy()

# Feature Extraction
FEATURE_COLS = [col for col in df_cleaned.columns if col != target_col]
print(f"Number of features selected dynamically: {len(FEATURE_COLS)}")

# Enforce numeric types and handle infs
for col in FEATURE_COLS:
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

# KNN Imputation for acoustic integrity
print("Applying KNN Imputation...")
imputer = KNNImputer(n_neighbors=5, weights="distance")
X = imputer.fit_transform(df_cleaned[FEATURE_COLS])

y_label = df_cleaned[target_col].astype(str).str.strip().values

# Encode Labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_label)

Loading dataset from: dataset\all_emotions.csv
Number of features selected dynamically: 48
Applying KNN Imputation...


### 3. Pipeline Splitting & Scaling
We implement an 80/20 Stratified Train-Test Split. The `StandardScaler` is fitted *strictly* on the training set to prevent data leakage.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training Set: {X_train_scaled.shape[0]} samples")
print(f"Testing Set: {X_test_scaled.shape[0]} samples")

Training Set: 43588 samples
Testing Set: 10897 samples


### 4. Model Instantiation
We initialize the models. Early stopping configurations are set here or passed dynamically during the `.fit()` pipeline.

In [4]:
best_params = {}
if os.path.isfile("best_params.json"):
    with open("best_params.json", "r") as f:
        best_params = json.load(f)
    print("Loaded best_params.json successfully.")

xgb_params = best_params.get("xgboost", {
    "n_estimators": 468, "max_depth": 10, "learning_rate": 0.175, 
    "subsample": 0.969, "colsample_bytree": 0.772, "gamma": 1e-08
})
xgb_model = xgb.XGBClassifier(**xgb_params, random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss", objective="multi:softprob", num_class=len(encoder.classes_), device="cuda" if CUDA_AVAILABLE else "cpu", early_stopping_rounds=20)

lgb_params = best_params.get("lightgbm", {
    "n_estimators": 499, "max_depth": 11, "num_leaves": 67, 
    "learning_rate": 0.246, "subsample": 0.690, "colsample_bytree": 0.755
})
lgb_model = lgb.LGBMClassifier(**lgb_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, objective="multiclass", num_class=len(encoder.classes_), device="gpu" if CUDA_AVAILABLE else "cpu")

cb_params = best_params.get("catboost", {
    "iterations": 500, "depth": 8, "learning_rate": 0.15, "l2_leaf_reg": 3.0
})
cb_model = CatBoostClassifier(**cb_params, loss_function="MultiClass", random_seed=RANDOM_STATE, thread_count=-1, verbose=False, task_type="GPU" if CUDA_AVAILABLE else "CPU", early_stopping_rounds=20)
print("Models instantiated.")

Loaded best_params.json successfully.
Models instantiated.


### 5. Out-Of-Fold (OOF) Weight Optimization
We split the training data into 3 folds and utilize strict early stopping against the validation fold to avoid lockups. We then use SciPy's SLSQP minimizer to solve for the continuous optimal distribution of blending weights.

In [5]:
print("Training Out-Of-Fold models for weight optimization...")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

xgb_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))
lgb_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))
cb_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
    print(f"  Processing Fold {fold + 1}...")
    X_tr, y_tr = X_train_scaled[train_idx], y_train[train_idx]
    X_va, y_va = X_train_scaled[val_idx], y_train[val_idx]

    # Strict cloning to prevent parameter bleed between folds
    xgb_m = clone(xgb_model)
    lgb_m = clone(lgb_model)
    cb_m = clone(cb_model)

    # XGB and CatBoost use early_stopping_rounds defined in constructor
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    cb_m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
    
    # LightGBM dynamically handles early stopping callback
    try:
        lgb_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])
    except AttributeError:
        lgb_m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=20, verbose=False)

    xgb_oof[val_idx] = xgb_m.predict_proba(X_va)
    lgb_oof[val_idx] = lgb_m.predict_proba(X_va)
    cb_oof[val_idx] = cb_m.predict_proba(X_va)

# SLSQP Optimization for Continuous Weights
print("\nRunning SLSQP optimization for mathematically optimal blending weights...")
def objective_func(weights):
    w_xgb, w_lgb, w_cb = weights
    val_proba = (w_xgb * xgb_oof + w_lgb * lgb_oof + w_cb * cb_oof)
    preds = np.argmax(val_proba, axis=1)
    return -f1_score(y_train, preds, average="weighted")

constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
initial_weights = np.array([1.0, 1.0, 1.0]) / 3.0

res = minimize(objective_func, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = res.x

print(f"Optimized Blending Weights: XGBoost={best_weights[0]:.4f}, LightGBM={best_weights[1]:.4f}, CatBoost={best_weights[2]:.4f}")

Training Out-Of-Fold models for weight optimization...
  Processing Fold 1...
  Processing Fold 2...
  Processing Fold 3...

Running SLSQP optimization for mathematically optimal blending weights...
Optimized Blending Weights: XGBoost=0.3333, LightGBM=0.3333, CatBoost=0.3333


### 6. Final Training & Blind Test Evaluation
We fit the models one final time on the entire 80% training set. We generate raw probability predictions on the unseen 20% Test Set, apply our calculated optimal weights, and output the ultimate classification report.

In [6]:
print("\nFitting final models on full scaled training set...")
# Strip early stopping to allow full learning on the final composite training set
xgb_model.set_params(early_stopping_rounds=None)
cb_model.set_params(early_stopping_rounds=None)

xgb_model.fit(X_train_scaled, y_train, verbose=False)
lgb_model.fit(X_train_scaled, y_train, verbose=False)
cb_model.fit(X_train_scaled, y_train, verbose=False)

xgb_proba = xgb_model.predict_proba(X_test_scaled)
lgb_proba = lgb_model.predict_proba(X_test_scaled)
cb_proba = cb_model.predict_proba(X_test_scaled)

ensemble_proba = (best_weights[0] * xgb_proba + best_weights[1] * lgb_proba + best_weights[2] * cb_proba)
ensemble_pred = np.argmax(ensemble_proba, axis=1)

print("\n=================== ULTIMATE ENSEMBLE TEST REPORT ===================")
print(classification_report(y_test, ensemble_pred, target_names=encoder.classes_, digits=4))


Fitting final models on full scaled training set...


TypeError: LGBMClassifier.fit() got an unexpected keyword argument 'verbose'

### 7. Hard-Save Production Artifacts to Disk
We serialize the 3 trained models, the KNN imputer, the scaler, the label encoder, and the optimal weights.

In [ ]:
print("\nSerializing Production Artifacts for Live Backend...")

joblib.dump(xgb_model, 'ser_xgb_model.joblib')
joblib.dump(lgb_model, 'ser_lgb_model.joblib')
joblib.dump(cb_model,  'ser_cb_model.joblib')
joblib.dump(imputer,   'ser_knn_imputer.joblib')
joblib.dump(scaler,    'ser_ensemble_scaler.joblib')
joblib.dump(encoder,   'ser_ensemble_encoder.joblib')

weights_dict = {
    "xgb_weight": float(best_weights[0]),
    "lgb_weight": float(best_weights[1]),
    "cb_weight":  float(best_weights[2])
}
with open('ensemble_weights.json', 'w') as f:
    json.dump(weights_dict, f, indent=4)

print("SUCCESS: All Models, Imputer, Scaler, Encoder, and Weights safely flushed to disk!")
print("Ready for backend deployment.")